# Premier League Match Analysis

**Course:** PDS301M  
**Project:** Phân tích kết quả bóng đá Premier League

This notebook follows a reproducible path from the season CSV files saved in `data/raw/` to research-question tables, visualizations, findings, and limitations. Reusable loading, cleaning, feature engineering, and chart logic lives in `src/`; the notebook organizes and explains the analysis. The study is descriptive and covers only the seasons present in the local dataset.


## 1. Introduction

Premier League match records contain full-time and half-time scores, outcomes, and match statistics. We use these records to describe home advantage, team scoring and shooting, half-time to full-time outcomes, goal patterns, and relationships between match statistics. The questions come from project . We examine associations and observed patterns; this analysis reports observed associations and does not build a prediction model.


## 2. Research Questions

| ID | Question | Evidence used |
|---|---|---|
| RQ1 | Does home advantage appear in the analyzed matches? | Home-win, draw, and away-win counts and shares |
| RQ2 | Which teams scored and shot the most? | Combined home/away goals, shots, and shots on target |
| RQ3 | How is the half-time result associated with the full-time result? | Half-time to full-time count and row-percentage tables |
| RQ4 | How are total goals distributed across matches and seasons? | Summary statistics, histogram, and season box plot |
| RQ5 | How do match-statistic totals vary by result and relate to one another? | Outcome-group averages, scatter plot, and correlation heatmap |

The P6 wording asks whether half-time results can predict full-time outcomes. This notebook describes their association; a predictive claim would require a separately tested model and validation data.


## 3. Sources

- [Football-Data.co.uk — England match data and CSV downloads](https://www.football-data.co.uk/englandm.php): primary source for match results and available statistics.
- [Football-Data.co.uk — example Premier League 2025/26 CSV](https://www.football-data.co.uk/mmz4281/2526/E0.csv): example direct season file listed in P6.
- [Wikipedia — List of Premier League seasons](https://en.wikipedia.org/wiki/List_of_Premier_League_seasons): optional season context listed in P6; it is not used as match-level statistical data.

The notebook reads the CSV snapshots already saved under `data/raw/`. It does not make network requests or replace those files during **Restart Kernel → Run All**. Record the actual download date and source URL in your project report if your instructor asks for collection provenance.


## 4. Python Basics

The project helpers in `src/data_pipeline.py` demonstrate arithmetic, mapping a result code, tuple return values, and a conditional category. They are also reused in the analysis rather than being redefined in notebook cells.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Work when the kernel starts in either the repository root or notebooks/.
_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in _candidates if (p / "src").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the project repository so that src/ and data/ are available.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline import (
    total_goals, result_label, points_for_result, classify_match,
    unique_teams, team_appearances, match_key, load_raw_csvs,
    inspect_data, clean_matches, create_features, summarize_by_outcome,
)
from src.analysis import generate_all_visualizations
from scripts.generate_visualizations import write_visualization_insights

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CHARTS_DIR = PROJECT_ROOT / "charts"
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {RAW_DIR}")


In [ ]:
example_goals = total_goals(2, 1)
example_result = "H"
print("Total goals:", example_goals)
print("Readable result:", result_label(example_result))
print("Points (home, away):", points_for_result(example_result))
print("Goal category:", classify_match(example_goals))


## 5. Python Data Structures

A match can be represented as a dictionary; a list can hold multiple match records. The helpers return a **set** of unique team names, a **dictionary** of appearance counts, and a **tuple** match key. This small example uses the same reusable functions as the data workflow.


In [ ]:
example_records = [
    {"Season": "2023/24", "Date": "2023-08-11", "HomeTeam": "Burnley", "AwayTeam": "Manchester City"},
    {"Season": "2023/24", "Date": "2023-08-12", "HomeTeam": "Arsenal", "AwayTeam": "Nottingham Forest"},
]
print("Unique teams:", sorted(unique_teams(example_records)))
print("Appearances:", team_appearances(example_records))
print("First match key:", match_key(example_records[0]))


## 6. Data Collection

P6 recommends downloading one to three seasons of Football-Data CSVs and saving them in the project. The next cell inventories the local snapshots, showing the file, inferred season, row count, and available columns. No live download is needed to rerun the notebook.


In [ ]:
raw_data, raw_files = load_raw_csvs(RAW_DIR)
file_inventory = []
for path in raw_files:
    frame = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    season_value = frame["Season"].dropna().iloc[0] if "Season" in frame and frame["Season"].notna().any() else path.stem
    file_inventory.append({"File": path.name, "Season": str(season_value), "Rows": len(frame), "Columns": len(frame.columns)})
display(pd.DataFrame(file_inventory))
print(f"Combined raw rows: {len(raw_data):,}")


## 7. Data Understanding

Before cleaning, inspect the shape, seasons, data types, missing values, incomplete rows, and duplicate match keys. Football-Data also publishes odds and optional match-statistic fields; missing optional values remain missing and are not treated as zero.


In [ ]:
audit = inspect_data(raw_data)
print(f"Raw shape: {audit['row_count']:,} rows × {audit['column_count']:,} columns")
print("Rows by season:")
display(audit["rows_by_season"].rename("Raw rows").to_frame())
print("Data types:")
display(raw_data.dtypes.rename("dtype").to_frame())
print("Missing values by column:")
display(audit["missing_by_column"].loc[lambda x: x.gt(0)].rename("Missing values").to_frame())
display(pd.DataFrame([{"Incomplete matches before cleaning": audit["incomplete_matches"], "Duplicate match keys before cleaning": audit["duplicate_matches"]}]))


## 8. Data Cleaning

The reusable cleaning function standardizes team-name whitespace, parses dates, converts score and available match-statistic columns to numeric values, retains completed matches with valid nonnegative full-time scores, derives a consistent full-time result, and removes duplicate fixtures. Missing optional statistics are preserved. Raw CSV snapshots remain unchanged.


In [ ]:
cleaned_data, cleaning_counts = clean_matches(raw_data)
cleaning_table = pd.DataFrame([cleaning_counts])
display(cleaning_table)
if cleaned_data.empty:
    raise ValueError("Cleaning left no completed matches. Check the CSV files and required columns.")
print(f"Cleaned completed matches: {len(cleaned_data):,}")


## 9. Feature Engineering

The reusable feature function adds total goals, goal difference, readable result labels, a high-scoring indicator (four or more goals), and combined shots, shots on target, and corners when both home and away columns are available.


In [ ]:
matches = create_features(cleaned_data)
show_columns = [c for c in ["Season", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "total_goals", "goal_difference", "result_label", "high_scoring", "total_shots", "total_shots_on_target", "total_corners"] if c in matches.columns]
display(matches[show_columns].head(10))
assert matches["total_goals"].equals(matches["FTHG"] + matches["FTAG"])
assert matches["goal_difference"].equals(matches["FTHG"] - matches["FTAG"])
assert matches["FTR"].isin(["H", "D", "A"]).all()
print(f"Feature-engineered matches: {len(matches):,}")


## 10. NumPy Analysis

NumPy summarizes total goals per match with descriptive measures before we interpret the Pandas tables. These are descriptive statistics for the completed matches in the local files.


In [ ]:
goal_values = matches["total_goals"].to_numpy(dtype=float)
numpy_goal_summary = pd.Series({
    "Mean goals": np.mean(goal_values),
    "Median goals": np.median(goal_values),
    "Standard deviation": np.std(goal_values, ddof=1) if len(goal_values) > 1 else 0.0,
    "90th percentile": np.percentile(goal_values, 90),
    "Matches with 4+ goals (%)": np.mean(goal_values >= 4) * 100,
}, name="Value")
display(numpy_goal_summary.to_frame().round(2))


## 11. Pandas Analysis — RQ1–RQ5

The tables answer each research question directly. All summaries use the cleaned, completed match rows created above.


### RQ1 — Does home advantage appear?

Compare the home-win share with the draw and away-win shares. A larger home-win share is descriptive evidence of a home advantage in this sample, not evidence of directionality.


In [ ]:
outcome_summary = summarize_by_outcome(matches)
outcome_summary.index = ["Home win", "Draw", "Away win"]
display(outcome_summary.round(1))


### RQ2 — Which teams scored and shot the most?

Home and away records are combined so each team's goals and available shots are counted regardless of venue. The table includes appearance-normalized rates for fairer comparisons.


In [ ]:
home_attack = pd.DataFrame({"Team": matches["HomeTeam"], "Goals": matches["FTHG"]})
away_attack = pd.DataFrame({"Team": matches["AwayTeam"], "Goals": matches["FTAG"]})
for stat, home_col, away_col in [("Shots", "HS", "AS"), ("Shots on target", "HST", "AST")]:
    if home_col in matches and away_col in matches:
        home_attack[stat] = matches[home_col]
        away_attack[stat] = matches[away_col]
team_attack = pd.concat([home_attack, away_attack], ignore_index=True)
team_attack_summary = team_attack.groupby("Team").agg(Appearances=("Team", "size"))
team_attack_summary["Goals"] = team_attack.groupby("Team")["Goals"].sum(min_count=1)
for stat in ["Shots", "Shots on target"]:
    if stat in team_attack:
        team_attack_summary[stat] = team_attack.groupby("Team")[stat].sum(min_count=1)
team_attack_summary["Goals per appearance"] = team_attack_summary["Goals"] / team_attack_summary["Appearances"]
for stat in ["Shots", "Shots on target"]:
    if stat in team_attack_summary:
        team_attack_summary[f"{stat} per appearance"] = team_attack_summary[stat] / team_attack_summary["Appearances"]
team_attack_summary = team_attack_summary.sort_values(["Goals", "Goals per appearance"], ascending=False)
display(team_attack_summary.head(10).round(2))


### RQ3 — How is the half-time result associated with the full-time result?

The count table shows how matches moved from half-time to full-time. Row percentages show the final-result mix among games with the same half-time result; they describe association and are not forecast accuracy.


In [ ]:
result_order = ["H", "D", "A"]
result_labels = {"H": "Home win", "D": "Draw", "A": "Away win"}
if "HTR" not in matches:
    matches["HTR"] = pd.NA
ht_counts = pd.crosstab(matches["HTR"], matches["FTR"]).reindex(index=result_order, columns=result_order, fill_value=0)
ht_counts.index = [result_labels[x] for x in result_order]
ht_counts.columns = [result_labels[x] for x in result_order]
print("Match counts:")
display(ht_counts)
print("Row percentages:")
display((ht_counts.div(ht_counts.sum(axis=1).replace(0, np.nan), axis=0) * 100).round(1))


### RQ4 — How are total goals distributed?

Summarize the number of goals per match overall and by season. The histogram and box plot in the Visualization section show the distribution and variation.


In [ ]:
rq4 = matches["total_goals"].describe().rename("Total goals").to_frame()
rq4.loc["Matches with 4+ goals (%)", "Total goals"] = matches["high_scoring"].mean() * 100
display(rq4.round(2))
if "Season" in matches:
    display(matches.groupby("Season")["total_goals"].agg(Matches="count", Mean="mean", Median="median", Std="std").round(2))


### RQ5 — How do match statistics vary by result and relate to one another?

We compare average combined match statistics across outcomes and calculate pairwise Pearson correlations. A correlation describes a linear association; it does not establish directionality between the statistic and match outcome. Missing values are excluded for the statistic being summarized.


In [ ]:
available_totals = [c for c in ["total_shots", "total_shots_on_target", "total_corners"] if c in matches and matches[c].notna().any()]
rq5 = matches[["FTR", *available_totals]].copy()
rq5["Outcome"] = rq5["FTR"].map(result_labels)
if available_totals:
    rq5_by_outcome = rq5.groupby("Outcome")[available_totals].mean().reindex(["Home win", "Draw", "Away win"])
    print("Mean match-statistic totals by full-time result:")
    display(rq5_by_outcome.round(2))
    print("Pearson correlations among available match-statistic totals:")
    display(matches[available_totals].corr().round(2))
else:
    print("No paired shots, shots-on-target, or corners columns were available in these CSVs.")


## 12. Visualization

The reusable chart functions in `src/analysis.py` generate eight figures under `charts/`: outcome bar chart, top-team bar chart, half-time/full-time heatmap, goal histogram, season box plot, match-statistic correlation heatmap, goals-versus-shots scatter plot, and season comparison bar chart. Each chart includes a short evidence-based insight; the same insights are written to `charts/insights.md`.


In [ ]:
visualization_summaries = generate_all_visualizations(matches, CHARTS_DIR)
insights_path = write_visualization_insights(matches, visualization_summaries, CHARTS_DIR)
chart_files = sorted(CHARTS_DIR.glob("*.png"))
assert len(chart_files) >= 5, "Expected the assignment-required chart types to be generated."
print(f"Generated {len(chart_files)} charts and {insights_path.name} in {CHARTS_DIR}")
for chart_path in chart_files:
    display(Markdown(f"#### {chart_path.name}"))
    display(Image(filename=str(chart_path)))


## 13. Findings

The statements below are generated from the same cleaned rows and chart summaries shown above. Review the values alongside the tables and figures; they describe the local season files only.


In [ ]:
home_rate = float(visualization_summaries["home_advantage"].get("Home win", 0))
away_rate = float(visualization_summaries["home_advantage"].get("Away win", 0))
top_team = visualization_summaries["team_performance"].index[0]
top_team_goals = int(visualization_summaries["team_performance"].iloc[0])
ht_table = visualization_summaries["halftime_fulltime"]
transition = ht_table.stack().idxmax() if ht_table.to_numpy().sum() else ("No half-time data", "No full-time data")
transition_count = int(ht_table.loc[transition[0], transition[1]]) if ht_table.to_numpy().sum() else 0
goal_mean = float(matches["total_goals"].mean())
goal_median = float(matches["total_goals"].median())
season_summary = visualization_summaries["season_comparison"]
top_season = season_summary.sort_values("MeanGoals", ascending=False).iloc[0]
findings = [
    {"Research question": "RQ1 — Home advantage", "Evidence": f"Home wins were {home_rate:.1f}% of matches; away wins were {away_rate:.1f}%."},
    {"Research question": "RQ2 — Team attack", "Evidence": f"{top_team} scored the most goals ({top_team_goals:,}) in the included seasons."},
    {"Research question": "RQ3 — Half-time to full-time", "Evidence": f"Most common transition: {transition[0]} → {transition[1]} ({transition_count:,} matches)."},
    {"Research question": "RQ4 — Goal distribution", "Evidence": f"Mean total goals were {goal_mean:.2f}; median was {goal_median:.1f} per match."},
    {"Research question": "RQ5 — Match statistics", "Evidence": f"See outcome averages and correlations above; {top_season['Season']} had the highest seasonal mean ({top_season['MeanGoals']:.2f} goals per match)."},
]
display(pd.DataFrame(findings))


## 14. Limitations

- The sample includes only completed matches and only the season CSV files saved in `data/raw/`; it may not represent every Premier League season.
- Optional columns such as odds and some match statistics can be missing or incomplete. The cleaning code preserves missing values and excludes them from the affected calculation.
- Team totals are affected by the number of appearances; per-appearance rates should be considered when comparing teams.
- Correlation and outcome comparisons are descriptive and do not establish directionality. Half-time/full-time association is not a validated prediction model.
- Team names and source snapshots can change between downloads. Results may differ if the local CSV files are refreshed.


## 15. Conclusion

This analysis answers the P6 questions with outcome shares, team summaries, half-time/full-time comparisons, goal-distribution measures, match-statistic comparisons, and visualizations. Results are calculated from the cleaned local CSV data; the notebook keeps reusable loading, cleaning, features, and plotting logic in `src/` and records evidence for each research question. Interpret the findings as patterns in the analyzed matches, as descriptive patterns rather than directional effects or prediction guarantees.
